# WildSpeech P01 — Literacy Signal Discovery

**Goal.** Find out whether children's speech recordings carry information about end-of-year literacy (`outcome_score`) **beyond** what the beginning-of-year screening already tells us (`*_theta`), and do it honestly: student-level targets, student- and school-disjoint validation, multiple-testing control, confound checks, and an explicit hypothesis scorecard.

**How to use.** Edit the parameters cell, then *Run All*. Everything is plain Python in this notebook — no config files, no pipeline framework. Audio features are extracted once (streamed straight from the `.tar` archives, nothing is unpacked) and cached as Parquet in `WildSpeech_outputs/`; later runs reload them.

| Step | Question |
|---|---|
| 1 | Is the data consistent across student / task / item / clip levels? |
| 2 | What does the outcome look like (instrument, school effects)? |
| 3 | How much does screening explain? |
| 4 | Which items are informative? What do responses (accuracy, Rasch ability, items reached) add? |
| 5 | Acoustic extraction: quality, temporal/fluency, spectral, MFCC, prosody, wavelet |
| 6 | Which clip-level acoustics relate to correctness *within the same item*? |
| 7 | Student-level aggregation (item-adjusted) |
| 8 | Honest CV baselines → screening residuals |
| 9 | Residual signal discovery (FDR, bootstrap, split-half replication) |
| 10 | Confounds: sample rate, school, technical quality, negative controls |
| 11 | Student phenotypes (clustering) |
| 12 | Predictive ablation by feature family (paired bootstrap deltas) |
| 13 | Failure atlas |
| 14 | Feature registry + task relevance matrix |
| 15 | Hypothesis registry |
| 16 | Submission |
| 17 | Summary |

In [ ]:
# ======================= PARAMETERS — edit here =======================
from pathlib import Path

DATA_DIR = Path("data")                  # CSVs + train_audio.tar + test_audio.tar
OUT_DIR = Path("WildSpeech_outputs")     # tables, figures, cached features, submission

SEED = 42
N_JOBS = -1                  # CPU workers for audio feature extraction (-1 = all cores)
SR = 16000                   # target sample rate (clips are stored at 16 kHz)
MAX_CLIPS_PER_SPLIT = None   # e.g. 600 for a quick smoke run; None = every clip
REBUILD_FEATURES = False     # True = re-extract audio features even if a cache exists
EXTRACT_TEST_AUDIO = True    # needed for the submission

# voice-activity / pause definitions (seconds)
MIN_SEGMENT = 0.05
MERGE_GAP = 0.08
SHORT_PAUSE = 0.15
LONG_PAUSE = 0.50

# validation / statistics
N_FOLDS = 5
N_REPEATS = 3
N_BOOT = 1000
FDR_ALPHA = 0.05
TOP_K_FEATURES = 40          # in-fold univariate selection for the "ridge_topk" model
CONFOUND_R = 0.30            # |r| with sample-rate / instrument above which a feature is flagged
MIN_SCHOOL_SIZE = 10

MAKE_SUBMISSION = True

In [ ]:
import io, json, re, tarfile, time, warnings
from collections import OrderedDict

import numpy as np
import pandas as pd
import scipy.ndimage as ndi
from scipy import stats
import soundfile as sf
import librosa
import pywt
import matplotlib
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from IPython.display import display, Markdown

from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.dummy import DummyRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.metrics import mean_squared_error, r2_score, silhouette_score, roc_auc_score
from sklearn.model_selection import GroupKFold, RepeatedKFold, StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMRegressor
import scipy.sparse as sp

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50, "display.width", 200, "display.float_format", "{:.4g}".format)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

if not DATA_DIR.exists() and (Path("..") / DATA_DIR).exists():
    DATA_DIR = Path("..") / DATA_DIR
assert (DATA_DIR / "train_labels.csv").exists(), f"data not found in {DATA_DIR.resolve()}"
FIG_DIR, TAB_DIR = OUT_DIR / "figures", OUT_DIR / "tables"
for d in (OUT_DIR, FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEED)


def save_table(df, name, index=False):
    df.to_csv(TAB_DIR / f"{name}.csv", index=index)
    return df


def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def bh_fdr(p):
    """Benjamini-Hochberg q-values (NaN-safe)."""
    p = np.asarray(p, float)
    q = np.full_like(p, np.nan)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return q
    pv = p[ok]
    order = np.argsort(pv)
    ranked = pv[order] * ok.sum() / (np.arange(ok.sum()) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty_like(pv)
    out[order] = np.minimum(ranked, 1)
    q[ok] = out
    return q


def spearman(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < 20 or np.nanstd(x[ok]) == 0:
        return np.nan, np.nan, int(ok.sum())
    r, p = stats.spearmanr(x[ok], y[ok])
    return float(r), float(p), int(ok.sum())


print("data   :", DATA_DIR.resolve())
print("outputs:", OUT_DIR.resolve())
print("librosa", librosa.__version__, "| pandas", pd.__version__)

## 1 · Data integrity and hierarchy

Levels: **student** (metadata, screening thetas, label) → **task** (blending, deletion, sentence repetition) → **item** (`expected_text`) → **clip** (one audio response). Checks: key uniqueness, joins, train/test disjointness, scoring status, archive ↔ metadata match, and whether a missing theta means the task was never administered.

In [ ]:
clips_tr = pd.read_csv(DATA_DIR / "train_clip_metadata.csv")
clips_te = pd.read_csv(DATA_DIR / "test_clip_metadata.csv")
stud_tr = pd.read_csv(DATA_DIR / "train_student_metadata.csv")
stud_te = pd.read_csv(DATA_DIR / "test_student_metadata.csv")
labels = pd.read_csv(DATA_DIR / "train_labels.csv")
sub_fmt = pd.read_csv(DATA_DIR / "submission_format.csv")

clips_tr["split"], clips_te["split"] = "train", "test"
clips = pd.concat([clips_tr, clips_te], ignore_index=True)
TASKS = ["blending", "deletion", "sentence_repetition"]
TASK_ABBR = {"blending": "bl", "deletion": "de", "sentence_repetition": "sr"}
THETAS = ["blending_theta", "deletion_theta", "sentence_repetition_theta"]


def tar_names(path):
    with tarfile.open(path) as tf:
        return {Path(m.name).name for m in tf.getmembers() if m.isfile()}


tar_tr, tar_te = tar_names(DATA_DIR / "train_audio.tar"), tar_names(DATA_DIR / "test_audio.tar")

checks = OrderedDict()
checks["train students unique"] = stud_tr.subject_id.is_unique
checks["test students unique"] = stud_te.subject_id.is_unique
checks["clip filenames unique"] = clips.filename.is_unique
checks["every train student has a label"] = set(stud_tr.subject_id) == set(labels.subject_id)
checks["train clips -> known train students"] = clips_tr.subject_id.isin(stud_tr.subject_id).all()
checks["test clips -> known test students"] = clips_te.subject_id.isin(stud_te.subject_id).all()
checks["train/test students disjoint"] = len(set(stud_tr.subject_id) & set(stud_te.subject_id)) == 0
checks["submission ids == test students"] = set(sub_fmt.subject_id) == set(stud_te.subject_id)
checks["train clips all in train tar"] = clips_tr.filename.isin(tar_tr).all()
checks["test clips all in test tar"] = clips_te.filename.isin(tar_te).all()
checks["no extra files in tars"] = (len(tar_tr - set(clips_tr.filename)) + len(tar_te - set(clips_te.filename))) == 0
checks["tasks are the expected three"] = set(clips.task) == set(TASKS)
display(pd.DataFrame({"check": list(checks), "ok": list(checks.values())}))
assert all(checks.values()), "integrity failure — see table above"

students = pd.concat([stud_tr.merge(labels, on="subject_id").assign(split="train"),
                      stud_te.assign(split="test")], ignore_index=True)
students["wj4"] = (students.instrument == "wj4").astype(int)
students["sr_11025"] = (students.original_sample_rate == 11025).astype(int)

# does a missing theta mean "task not administered"?
has_task = clips.groupby(["subject_id", "task"]).size().unstack(fill_value=0).reindex(columns=TASKS, fill_value=0) > 0
rows = []
for t, th in zip(TASKS, THETAS):
    s = students.set_index("subject_id")
    ht = has_task[t].reindex(s.index, fill_value=False)
    rows.append({"task": t, "theta missing": int(s[th].isna().sum()),
                 "missing theta & no clips": int((s[th].isna() & ~ht).sum()),
                 "missing theta & has clips": int((s[th].isna() & ht).sum()),
                 "theta present & no clips": int((s[th].notna() & ~ht).sum())})
display(pd.DataFrame(rows))

print("score_status (train):")
display(clips_tr.score_status.value_counts().to_frame())
hier = (clips.groupby(["split", "task"]).agg(clips=("filename", "size"), students=("subject_id", "nunique"),
                                          items=("expected_text", "nunique"), accuracy=("item_score", "mean"),
                                          median_dur=("duration_sec", "median")).reset_index())
display(save_table(hier, "01_hierarchy"))
print("schools in train:", stud_tr.school_id.nunique(), "| test has school_id:", "school_id" in stud_te)
print("original sample rate:", students.groupby(["split", "original_sample_rate"]).size().to_dict())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
cps = clips.groupby(["split", "subject_id"]).size()
for sp_ in ["train", "test"]:
    ax[0].hist(cps[sp_], bins=40, alpha=.6, label=sp_)
ax[0].set(title="clips per student", xlabel="clips"); ax[0].legend()
for t in TASKS:
    ax[1].hist(np.log10(clips.loc[clips.task == t, "duration_sec"].clip(0.05)), bins=50, alpha=.5, label=t)
ax[1].set(title="clip duration", xlabel="log10 seconds"); ax[1].legend(fontsize=8)
pd.crosstab(students.loc[students.split == "train", "school_id"], students.loc[students.split == "train", "instrument"]).plot.bar(stacked=True, ax=ax[2])
ax[2].set(title="train students per school × instrument", xlabel="school")
save_fig(fig, "01_hierarchy")

## 2 · Target intelligence

The outcome comes from two different instruments (KTEA-3, WJ-IV) — they are compared, never assumed interchangeable. School effects are quantified (ICC) because they can masquerade as signal.

In [ ]:
tr = students[students.split == "train"].reset_index(drop=True)
y = tr.outcome_score.to_numpy(float)

summ = tr.groupby("instrument").outcome_score.describe()
summ.loc["all"] = tr.outcome_score.describe()
summ["skew"] = [stats.skew(tr.outcome_score[tr.instrument == i]) for i in summ.index[:-1]] + [stats.skew(y)]
display(save_table(summ.reset_index(), "02_target_summary"))

mw = stats.mannwhitneyu(*[tr.outcome_score[tr.instrument == i] for i in ["ktea3", "wj4"]])
lev = stats.levene(*[tr.outcome_score[tr.instrument == i] for i in ["ktea3", "wj4"]])
print(f"instrument location diff: Mann-Whitney p={mw.pvalue:.3g} | spread diff: Levene p={lev.pvalue:.3g}")


def icc_oneway(values, groups):
    df = pd.DataFrame({"v": values, "g": groups}).dropna()
    k = df.groupby("g").size(); k = k[k >= 2]; df = df[df.g.isin(k.index)]
    grand = df.v.mean(); n = len(df); a = len(k)
    msb = (k * (df.groupby("g").v.mean() - grand) ** 2).sum() / (a - 1)
    msw = ((df.v - df.groupby("g").v.transform("mean")) ** 2).sum() / (n - a)
    k0 = (n - (k ** 2).sum() / n) / (a - 1)
    return max(0.0, (msb - msw) / (msb + (k0 - 1) * msw))


ICC_SCHOOL = icc_oneway(tr.outcome_score, tr.school_id)
print(f"school ICC of outcome = {ICC_SCHOOL:.3f}")
z = np.abs(stats.zscore(y))
print(f"|z|>3 outliers: {(z > 3).sum()} | floor/ceiling share (min/max value): {(y == y.min()).mean():.3f}/{(y == y.max()).mean():.3f}")

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for i in ["ktea3", "wj4"]:
    ax[0].hist(tr.outcome_score[tr.instrument == i], bins=30, alpha=.6, label=i)
ax[0].set(title="outcome by instrument", xlabel="outcome_score"); ax[0].legend()
order = tr.groupby("school_id").outcome_score.median().sort_values().index
ax[1].boxplot([tr.outcome_score[tr.school_id == s] for s in order], tick_labels=order)
ax[1].set(title=f"outcome by school (ICC={ICC_SCHOOL:.2f})", xlabel="school")
stats.probplot(y, plot=ax[2]); ax[2].set_title("normal QQ")
save_fig(fig, "02_target")

## 3 · Screening intelligence

How much of the outcome does the beginning-of-year screening explain? This is the bar every new signal must clear. Missingness is examined too — a task that was not administered may itself be informative.

In [ ]:
rows = []
for th in THETAS:
    r, p, n = spearman(tr[th], y)
    pr = stats.pearsonr(*tr[[th, "outcome_score"]].dropna().to_numpy().T)
    rows.append({"feature": th, "n": n, "pearson": pr[0], "spearman": r, "p": p,
                 "pearson_ktea3": tr.loc[tr.instrument == "ktea3", [th, "outcome_score"]].corr().iloc[0, 1],
                 "pearson_wj4": tr.loc[tr.instrument == "wj4", [th, "outcome_score"]].corr().iloc[0, 1]})
for th in THETAS:
    miss = tr[th].isna()
    rows.append({"feature": th + "_missing", "n": len(tr), "pearson": np.corrcoef(miss, y)[0, 1],
                 "spearman": spearman(miss.astype(float), y)[0], "p": stats.mannwhitneyu(y[miss], y[~miss]).pvalue if miss.any() else np.nan})
screen_assoc = save_table(pd.DataFrame(rows), "03_screening_associations")
display(screen_assoc)
display(tr[THETAS + ["outcome_score"]].corr().round(3))

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for a, th in zip(ax, THETAS):
    for i, c in [("ktea3", "C0"), ("wj4", "C1")]:
        d = tr[tr.instrument == i]
        a.scatter(d[th], d.outcome_score, s=8, alpha=.5, c=c, label=i)
    a.set(xlabel=th, ylabel="outcome_score")
ax[0].legend()
save_fig(fig, "03_screening_scatter")

## 4 · Item intelligence and response features

The assessment is adaptive, so raw accuracy is misleading (stronger children reach harder items). We estimate item difficulty with a regularised **Rasch** model per task (fitted on train + test *responses* — no labels involved), then build student response features:

* `r_{task}_n` items attempted, `r_{task}_acc` raw accuracy, `r_{task}_adj_acc` accuracy minus item expected-correct,
* `r_{task}_rasch` Rasch ability, `r_{task}_mean_difficulty` mean difficulty of items reached (adaptive routing signal),
* simple linguistic complexity of items (`words`, `letters`, approximate syllables).

Item–outcome associations below use all training students and are **EDA only** (not used to build predictors).

Response **order** (streaks, recovery, early/late accuracy) cannot be analysed: the data has no timestamps or sequence index, and dataframe order is not trusted.

In [ ]:
scored = clips[clips.item_score.notna()].copy()


def syllables(text):
    words = re.findall(r"[a-z]+", str(text).lower())
    return sum(max(1, len(re.findall(r"[aeiouy]+", w))) for w in words)


def fit_rasch(df, C=1.0):
    """Joint-ML Rasch with L2: logit P(correct) = ability_person - difficulty_item."""
    pid = pd.Index(df.subject_id.unique()); iid = pd.Index(df.expected_text.unique())
    pi, ii = pid.get_indexer(df.subject_id), iid.get_indexer(df.expected_text)
    n = len(df)
    X = sp.hstack([sp.csr_matrix((np.ones(n), (np.arange(n), pi)), shape=(n, len(pid))),
                   sp.csr_matrix((-np.ones(n), (np.arange(n), ii)), shape=(n, len(iid)))]).tocsr()
    m = LogisticRegression(C=C, max_iter=2000).fit(X, df.item_score.astype(int))
    w = m.coef_[0]
    ability = pd.Series(w[:len(pid)] + m.intercept_[0], index=pid)
    difficulty = pd.Series(w[len(pid):], index=iid)
    return ability, difficulty


abilities, items = {}, []
for t in TASKS:
    d = scored[scored.task == t]
    ab, dif = fit_rasch(d)
    abilities[t] = ab
    g = d.groupby("expected_text")
    it = pd.DataFrame({"task": t, "n_exposure": g.size(), "p_correct": g.item_score.mean(),
                       "p_smoothed": (g.item_score.sum() + 1) / (g.size() + 2), "rasch_difficulty": dif})
    # rest-score discrimination: item score vs the student's ability (point-biserial)
    d = d.assign(ability=d.subject_id.map(ab))
    it["discrimination"] = d.groupby("expected_text").apply(lambda x: x.item_score.corr(x.ability) if x.item_score.nunique() > 1 and len(x) >= 20 else np.nan)
    items.append(it)
items = pd.concat(items).reset_index().rename(columns={"index": "expected_text"})
items["n_words"] = items.expected_text.str.split().str.len()
items["n_letters"] = items.expected_text.str.count(r"[A-Za-z]")
items["n_syllables"] = items.expected_text.map(syllables)

# item vs outcome (EDA only, train students who saw the item)
lab = tr.set_index("subject_id").outcome_score
sc_tr = scored[scored.split == "train"].assign(outcome=lambda d: d.subject_id.map(lab))
io_assoc = sc_tr.groupby(["task", "expected_text"]).apply(
    lambda x: x.item_score.corr(x.outcome) if len(x) >= 20 and x.item_score.nunique() > 1 else np.nan).rename("r_outcome_eda")
items = items.merge(io_assoc.reset_index(), on=["task", "expected_text"], how="left")
save_table(items, "04_item_registry")

print("item difficulty vs complexity (spearman):")
display(items.groupby("task").apply(lambda d: pd.Series({c: spearman(d[c], d.rasch_difficulty)[0] for c in ["n_words", "n_letters", "n_syllables", "n_exposure"]})))
print("most outcome-informative items (EDA):")
display(items.dropna(subset=["r_outcome_eda"]).sort_values("r_outcome_eda", ascending=False).groupby("task").head(4)[["task", "expected_text", "n_exposure", "p_correct", "discrimination", "r_outcome_eda"]])

In [ ]:
# student response features
difficulty_map = items.set_index(["task", "expected_text"]).rasch_difficulty
p_map = items.set_index(["task", "expected_text"]).p_smoothed
scored["item_p"] = [p_map[(t, e)] for t, e in zip(scored.task, scored.expected_text)]
scored["item_diff"] = [difficulty_map[(t, e)] for t, e in zip(scored.task, scored.expected_text)]
scored["resid_score"] = scored.item_score - scored.item_p
syl = items.set_index(["task", "expected_text"]).n_syllables

resp = pd.DataFrame(index=students.subject_id)
for t in TASKS:
    a = TASK_ABBR[t]; d = scored[scored.task == t]; g = d.groupby("subject_id")
    resp[f"r_{a}_n"] = g.size()
    resp[f"r_{a}_acc"] = g.item_score.mean()
    resp[f"r_{a}_adj_acc"] = g.resid_score.mean()
    resp[f"r_{a}_rasch"] = abilities[t]
    resp[f"r_{a}_mean_difficulty"] = g.item_diff.mean()
    resp[f"r_{a}_max_difficulty_correct"] = d[d.item_score == 1].groupby("subject_id").item_diff.max()
    resp[f"r_{a}_mean_syllables"] = d.assign(s=[syl[(t, e)] for e in d.expected_text]).groupby("subject_id").s.mean()
resp[[f"r_{TASK_ABBR[t]}_n" for t in TASKS]] = resp[[f"r_{TASK_ABBR[t]}_n" for t in TASKS]].fillna(0)
resp["r_total_n"] = resp[[f"r_{TASK_ABBR[t]}_n" for t in TASKS]].sum(1)
resp["r_all_adj_acc"] = scored.groupby("subject_id").resid_score.mean()
resp["r_n_unscored"] = clips[clips.item_score.isna()].groupby("subject_id").size().reindex(resp.index).fillna(0)
resp = resp.reset_index().rename(columns={"index": "subject_id"})

cmp = resp.merge(students[["subject_id"] + THETAS], on="subject_id")
print("Rasch ability vs provided theta (pearson) — are the thetas built from these same responses?")
display(pd.Series({t: cmp[f"r_{TASK_ABBR[t]}_rasch"].corr(cmp[th]) for t, th in zip(TASKS, THETAS)}).to_frame("r"))

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for t in TASKS:
    d = items[items.task == t]
    ax[0].scatter(d.rasch_difficulty, d.n_exposure, s=12, label=t, alpha=.7)
    ax[1].scatter(d.rasch_difficulty, d.discrimination, s=12, alpha=.7)
ax[0].set(xlabel="Rasch difficulty", ylabel="exposures", title="adaptive exposure"); ax[0].legend(fontsize=8)
ax[1].set(xlabel="Rasch difficulty", ylabel="discrimination", title="item discrimination")
rt = resp.merge(tr[["subject_id", "outcome_score"]])
ax[2].scatter(rt.r_bl_mean_difficulty, rt.outcome_score, s=8, alpha=.5)
ax[2].set(xlabel="blending: mean difficulty of items reached", ylabel="outcome", title="adaptive routing vs outcome")
save_fig(fig, "04_items")

## 5 · Acoustic extraction

Each clip is decoded once and every family is computed from a shared STFT / energy envelope:

| prefix | family | content |
|---|---|---|
| `q_` | quality / technical | duration, peak, clipping, noise floor, SNR, effective bandwidth edge |
| `t_` | temporal / fluency | energy VAD → speech ratio, segments, initial silence (≠ verified reaction time), pauses |
| `s_` | spectral | centroid, bandwidth, roll-off, flatness, flux, ZCR, >4 kHz energy (speech frames) |
| `m_` | MFCC | 13 MFCC means / SDs / mean \|Δ\| on speech frames |
| `p_` | prosody | YIN pitch level / range / slope / jumps, intensity variation |
| `w_` | wavelet | db4 DWT energy fractions per scale, entropy, transient rate |

VAD marks *speech-like* regions, not specifically the child (a proctor may talk). Voice conversion anonymisation alters absolute pitch/timbre; §10 checks which features look technical rather than behavioural.

In [ ]:
FRAME, HOP, NFFT = 400, 160, 512           # 25 ms window, 10 ms hop at 16 kHz


def mask_to_segments(mask, hop_s, min_seg=MIN_SEGMENT, merge_gap=MERGE_GAP):
    m = np.concatenate([[0], mask.astype(np.int8), [0]])
    starts = np.flatnonzero(np.diff(m) == 1) * hop_s
    ends = np.flatnonzero(np.diff(m) == -1) * hop_s
    segs = []
    for s, e in zip(starts, ends):
        if segs and s - segs[-1][1] < merge_gap:
            segs[-1][1] = e
        else:
            segs.append([s, e])
    return [(s, e) for s, e in segs if e - s >= min_seg]


def clip_features(y, sr):
    f = {"q_duration": len(y) / sr}
    if len(y) < int(0.2 * sr) or not np.isfinite(y).all():
        return f
    y = (y - y.mean()).astype(np.float32)
    dur = len(y) / sr
    a = np.abs(y)
    f["q_peak"] = float(a.max())
    f["q_clip_ratio"] = float(np.mean(a > 0.98))
    rms = librosa.feature.rms(y=y, frame_length=FRAME, hop_length=HOP)[0]
    db = 20 * np.log10(rms + 1e-8)
    lo, hi = np.percentile(db, 10), np.percentile(db, 95)
    f["q_noise_floor_db"], f["q_level_db"], f["q_snr_db"] = lo, hi, hi - lo
    thr = lo + 0.35 * (hi - lo) if hi - lo > 6 else hi - 3
    active = ndi.median_filter((db > thr).astype(np.uint8), size=5).astype(bool)
    hop_s = HOP / sr

    # ---- temporal / fluency
    segs = mask_to_segments(active, hop_s)
    if segs:
        st = np.array([s for s, _ in segs]); en = np.array([e for _, e in segs])
        seg_d = en - st; gaps = st[1:] - en[:-1]; pauses = gaps[gaps >= SHORT_PAUSE]
        span = max(en[-1] - st[0], 1e-3); speech = seg_d.sum()
        f.update(t_speech_dur=speech, t_speech_ratio=speech / dur, t_n_segments=len(segs),
                 t_initial_silence=st[0], t_final_silence=dur - en[-1], t_speech_span=span,
                 t_mean_seg_dur=seg_d.mean(), t_max_seg_dur=seg_d.max(),
                 t_n_pauses=len(pauses), t_pause_total=pauses.sum(),
                 t_mean_pause=pauses.mean() if len(pauses) else 0.0,
                 t_max_pause=pauses.max() if len(pauses) else 0.0,
                 t_n_long_pauses=int((gaps >= LONG_PAUSE).sum()),
                 t_pause_ratio=pauses.sum() / span, t_segment_rate=len(segs) / max(speech, 1e-3))
    else:
        f.update(t_speech_dur=0.0, t_speech_ratio=0.0, t_n_segments=0)

    # ---- spectral (speech frames; fall back to the loudest 30 % of frames)
    S = np.abs(librosa.stft(y, n_fft=NFFT, hop_length=HOP, win_length=FRAME))
    nfr = min(S.shape[1], len(active)); S = S[:, :nfr]; act = active[:nfr]
    if act.sum() < 5:
        act = db[:nfr] >= np.percentile(db[:nfr], 70)
    Sa = S[:, act]
    freqs = librosa.fft_frequencies(sr=sr, n_fft=NFFT)
    P = Sa ** 2
    ltas = 10 * np.log10(P.mean(1) + 1e-12)
    above = np.flatnonzero(ltas > ltas.max() - 50)
    f["q_bandwidth_edge_hz"] = float(freqs[above.max()]) if len(above) else np.nan
    f["s_hf_ratio"] = float(P[freqs >= 4000].sum() / (P.sum() + 1e-12))
    spec = {"centroid": librosa.feature.spectral_centroid(S=Sa, sr=sr, n_fft=NFFT)[0],
            "bandwidth": librosa.feature.spectral_bandwidth(S=Sa, sr=sr, n_fft=NFFT)[0],
            "rolloff": librosa.feature.spectral_rolloff(S=Sa, sr=sr, n_fft=NFFT, roll_percent=0.85)[0],
            "flatness": librosa.feature.spectral_flatness(S=Sa)[0],
            "zcr": librosa.feature.zero_crossing_rate(y, frame_length=FRAME, hop_length=HOP)[0][:nfr][act]}
    if S.shape[1] > 1:
        flux = np.sqrt((np.diff(S, axis=1).clip(min=0) ** 2).sum(0))
        spec["flux"] = flux[act[1:]] if act[1:].any() else flux
    for k, v in spec.items():
        f[f"s_{k}_mean"], f[f"s_{k}_std"] = float(np.mean(v)), float(np.std(v))

    # ---- MFCC
    mel = librosa.feature.melspectrogram(S=S ** 2, sr=sr, n_mels=40)
    mf = librosa.feature.mfcc(S=librosa.power_to_db(mel + 1e-12), n_mfcc=13)
    mfa = mf[:, act]
    dmf = librosa.feature.delta(mf, width=9, mode="nearest")[:, act] if mf.shape[1] >= 9 else None
    for i in range(13):
        f[f"m_mfcc{i:02d}_mean"], f[f"m_mfcc{i:02d}_std"] = float(mfa[i].mean()), float(mfa[i].std())
        f[f"m_mfcc{i:02d}_dabs"] = float(np.abs(dmf[i]).mean()) if dmf is not None else np.nan

    # ---- prosody
    if len(y) >= 1024:
        f0 = librosa.yin(y, fmin=75, fmax=600, sr=sr, frame_length=1024, hop_length=HOP)[:nfr]
        vo = act[:len(f0)] & (f0 > 80) & (f0 < 570)
        f["p_voiced_ratio"] = float(vo.sum() / max(act.sum(), 1))
        if vo.sum() >= 10:
            st_ = 12 * np.log2(f0[vo] / 100.0)
            q5, q25, q50, q75, q95 = np.percentile(st_, [5, 25, 50, 75, 95])
            tt = np.flatnonzero(vo) * hop_s
            f.update(p_f0_median_st=q50, p_f0_iqr_st=q75 - q25, p_f0_range_st=q95 - q5,
                     p_f0_slope_st=float(np.polyfit(tt, st_, 1)[0]) if np.ptp(tt) > 0 else 0.0,
                     p_f0_jump_rate=float(np.mean(np.abs(np.diff(st_)) > 2)))
    ra = rms[:nfr][act]
    f["p_intensity_cv"] = float(ra.std() / (ra.mean() + 1e-9))

    # ---- wavelet
    lvl = min(6, pywt.dwt_max_level(len(y), 8))
    if lvl >= 3:
        coeffs = pywt.wavedec(y, "db4", level=lvl)
        en_ = np.array([np.sum(c.astype(np.float64) ** 2) for c in coeffs]); frac = en_ / (en_.sum() + 1e-12)
        names = ["A"] + [f"D{lvl - i}" for i in range(lvl)]
        for n_, v in zip(names, frac):
            f[f"w_energy_{n_}"] = float(v)
        f["w_entropy"] = float(-(frac * np.log(frac + 1e-12)).sum())
        d1 = coeffs[-1]; mad = np.median(np.abs(d1)) / 0.6745 + 1e-12
        f["w_transient_rate"] = float(np.sum(np.abs(d1) > 5 * mad) / dur)
        f["w_d2_kurtosis"] = float(stats.kurtosis(coeffs[-2]))
    return f


def process_batch(batch):
    out = []
    for name, raw in batch:
        try:
            y_, sr_ = sf.read(io.BytesIO(raw), dtype="float32", always_2d=False)
            if y_.ndim > 1:
                y_ = y_.mean(1)
            if sr_ != SR:
                y_ = librosa.resample(y_, orig_sr=sr_, target_sr=SR); sr_ = SR
            f = clip_features(y_, sr_); f["decode_ok"] = 1
        except Exception as e:
            f = {"decode_ok": 0, "decode_error": repr(e)[:200]}
        f["filename"] = name
        out.append(f)
    return out


def extract_split(split, wanted, batch_size=256):
    tag = "all" if MAX_CLIPS_PER_SPLIT is None else str(MAX_CLIPS_PER_SPLIT)
    cache = OUT_DIR / f"clip_features_{split}_{tag}.parquet"
    if cache.exists() and not REBUILD_FEATURES:
        print(f"[{split}] loading cached {cache.name}")
        return pd.read_parquet(cache)
    wanted = set(wanted)
    t0 = time.time(); results = []; batch = []; batches = []
    with tarfile.open(DATA_DIR / f"{split}_audio.tar") as tf, Parallel(n_jobs=N_JOBS) as par:
        for m in tf:
            name = Path(m.name).name
            if not m.isfile() or name not in wanted:
                continue
            batch.append((name, tf.extractfile(m).read()))
            if len(batch) == batch_size:
                batches.append(batch); batch = []
            if len(batches) == 32:
                for r in par(delayed(process_batch)(b) for b in batches):
                    results.extend(r)
                batches = []
                print(f"[{split}] {len(results)} clips, {time.time() - t0:.0f}s")
        if batch:
            batches.append(batch)
        for r in par(delayed(process_batch)(b) for b in batches):
            results.extend(r)
    df = pd.DataFrame(results)
    df.to_parquet(cache, index=False)
    print(f"[{split}] {len(df)} clips extracted in {time.time() - t0:.0f}s -> {cache.name}")
    return df


def pick(split_clips):
    if MAX_CLIPS_PER_SPLIT is None:
        return split_clips.filename
    return split_clips.sample(min(MAX_CLIPS_PER_SPLIT, len(split_clips)), random_state=SEED).filename


feat_tr = extract_split("train", pick(clips_tr))
feat_te = extract_split("test", pick(clips_te)) if EXTRACT_TEST_AUDIO else pd.DataFrame(columns=["filename"])
cf = clips.merge(pd.concat([feat_tr, feat_te], ignore_index=True), on="filename", how="inner")
FAMILIES = {"q": "quality", "t": "temporal", "s": "spectral", "m": "mfcc", "p": "prosody", "w": "wavelet"}
CLIP_FEATS = [c for c in cf.columns if c[:2] in {f"{k}_" for k in FAMILIES}]
print(f"clips with features: {len(cf)} | decode failures: {int((cf.decode_ok == 0).sum())} | clip features: {len(CLIP_FEATS)}")
print("features per family:", pd.Series([c[0] for c in CLIP_FEATS]).map(FAMILIES).value_counts().to_dict())

In [ ]:
# acoustic quality atlas
cf = cf.merge(students[["subject_id", "sr_11025", "instrument"]], on="subject_id", how="left")
qa = cf.groupby(["split", "sr_11025"])[["q_snr_db", "q_noise_floor_db", "q_bandwidth_edge_hz", "q_clip_ratio", "t_speech_ratio", "p_voiced_ratio"]].median()
display(save_table(qa.reset_index(), "05_quality_atlas"))
fig, ax = plt.subplots(1, 4, figsize=(16, 3.4))
for k, lab_ in [(0, "16 kHz"), (1, "11.025 kHz original")]:
    d = cf[cf.sr_11025 == k]
    ax[0].hist(d.q_snr_db, bins=60, alpha=.6, label=lab_, density=True)
    ax[1].hist(d.q_bandwidth_edge_hz, bins=60, alpha=.6, density=True)
    ax[2].hist(d.t_speech_ratio, bins=60, alpha=.6, density=True)
    ax[3].hist(d.t_initial_silence.clip(upper=10), bins=60, alpha=.6, density=True)
ax[0].set(title="SNR (dB)"); ax[1].set(title="effective bandwidth edge (Hz)")
ax[2].set(title="speech ratio"); ax[3].set(title="initial silence (s)"); ax[0].legend(fontsize=8)
save_fig(fig, "05_quality_atlas")

## 6 · Clip-level: acoustics vs correctness within the same item

Comparing correct vs incorrect clips across items mixes item content with performance. Here each feature is **centred within item** (`task × expected_text`), then related to correctness with a point-biserial correlation per task. Item ICC shows how much of each feature is item content.

In [ ]:
cfs = cf[cf.item_score.notna()].copy()
rows = []
for t in TASKS:
    d = cfs[cfs.task == t]
    for c in CLIP_FEATS:
        x = d[c]
        if x.notna().sum() < 100 or x.std() == 0:
            continue
        xc = x - d.groupby("expected_text")[c].transform("mean")
        ok = xc.notna()
        r, p = stats.pearsonr(xc[ok], d.item_score[ok])
        g = d[ok].groupby("expected_text")[c]
        icc = g.mean().var() / (x[ok].var() + 1e-12)
        rows.append({"task": t, "feature": c, "family": FAMILIES[c[0]], "r_within_item": r, "p": p, "item_var_share": min(icc, 1)})
within = pd.DataFrame(rows)
within["q"] = bh_fdr(within.p)
save_table(within, "06_within_item_correctness")
print("strongest within-item correctness associations per task:")
display(within.assign(a=within.r_within_item.abs()).sort_values("a", ascending=False).groupby("task").head(6).drop(columns="a"))

piv = within.pivot_table(index="family", columns="task", values="q", aggfunc=lambda q: (q < FDR_ALPHA).mean())
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
im = ax[0].imshow(piv.to_numpy(), cmap="viridis", vmin=0, vmax=1)
ax[0].set(xticks=range(len(piv.columns)), xticklabels=piv.columns, yticks=range(len(piv)), yticklabels=piv.index,
          title="share of features FDR-significant vs correctness")
for (i, j), v in np.ndenumerate(piv.to_numpy()):
    ax[0].text(j, i, f"{v:.2f}", ha="center", va="center", color="w")
for t in TASKS:
    d = within[within.task == t]
    ax[1].scatter(d.item_var_share, d.r_within_item.abs(), s=10, label=t, alpha=.7)
ax[1].set(xlabel="share of variance explained by item", ylabel="|r| with correctness (within item)"); ax[1].legend(fontsize=8)
save_fig(fig, "06_within_item")

## 7 · Student-level aggregation

The target is per student and clips are not independent examples. Children saw different items, so each clip feature is first **item-adjusted** (minus the item median over all train + test clips — no labels used), then summarised per student by the median overall (`all__`) and per task (`bl__`, `de__`, `sr__`). Missing tasks stay `NaN`; models impute inside each fold and get missingness indicators.

In [ ]:
item_med = cf.groupby(["task", "expected_text"])[CLIP_FEATS].transform("median")
adj = cf[CLIP_FEATS] - item_med
adj[["subject_id", "task"]] = cf[["subject_id", "task"]]

parts = [adj.groupby("subject_id")[CLIP_FEATS].median().add_prefix("all__")]
for t in TASKS:
    parts.append(adj[adj.task == t].groupby("subject_id")[CLIP_FEATS].median().add_prefix(f"{TASK_ABBR[t]}__"))
# behavioural contrast: how much longer / more hesitant are incorrect responses (within item)
for c in ["t_initial_silence", "t_speech_dur", "t_n_pauses", "q_duration"]:
    tmp = adj[[c, "subject_id"]].assign(score=cf.item_score)
    g = tmp.dropna().groupby(["subject_id", "score"])[c].median().unstack()
    if {0.0, 1.0} <= set(g.columns):
        parts.append((g[0.0] - g[1.0]).rename(f"all__{c}__wrong_minus_right").to_frame())
acoustic = pd.concat(parts, axis=1)
acoustic.index.name = "subject_id"
acoustic = acoustic.reset_index()

table = students.merge(resp, on="subject_id", how="left").merge(acoustic, on="subject_id", how="left")
SCREEN = THETAS + ["wj4", "sr_11025"]
RESP = [c for c in resp.columns if c.startswith("r_")]
ACOU = [c for c in acoustic.columns if c != "subject_id"]


def family_of(col):
    if col in SCREEN: return "screening"
    if col.startswith("r_"): return "response"
    base = col.split("__")[1]
    return FAMILIES[base[0]]


def task_of(col):
    if "__" not in col: return "student"
    return {"all": "all", "bl": "blending", "de": "deletion", "sr": "sentence_repetition"}[col.split("__")[0]]


TR = table[table.split == "train"].reset_index(drop=True)
TE = table[table.split == "test"].reset_index(drop=True)
y = TR.outcome_score.to_numpy(float)
schools = TR.school_id.to_numpy()
print(f"student table: train {TR.shape}, test {TE.shape}")
print("features by family:", pd.Series([family_of(c) for c in SCREEN + RESP + ACOU]).value_counts().to_dict())
print(f"train students with any audio features: {TR[ACOU].notna().any(axis=1).mean():.1%}")

## 8 · Validation framework and screening baselines

Two student-disjoint schemes, fixed once and reused by every model so comparisons are paired:

* **student** — repeated K-fold over students (`N_REPEATS × N_FOLDS`),
* **school** — grouped K-fold by `school_id` (new-school generalisation; the test set has no school id, so this is the pessimistic check).

All preprocessing (imputation, scaling, feature selection) lives inside the model pipeline, so it is refitted in every fold. The screening residual used for discovery is `outcome − OOF prediction` of the primary baseline (ridge on thetas + instrument + original sample rate).

In [ ]:
FOLDS = {
    "student": [(i // N_FOLDS, tr_i, te_i) for i, (tr_i, te_i) in
                enumerate(RepeatedKFold(n_splits=N_FOLDS, n_repeats=N_REPEATS, random_state=SEED).split(TR))],
    "school": [(0, tr_i, te_i) for tr_i, te_i in GroupKFold(n_splits=N_FOLDS).split(TR, groups=schools)],
}
for k, f in FOLDS.items():
    for _, a, b in f:
        assert not set(TR.subject_id.iloc[a]) & set(TR.subject_id.iloc[b])
    if k == "school":
        for _, a, b in f:
            assert not set(schools[a]) & set(schools[b])
print({k: len(v) for k, v in FOLDS.items()}, "folds — student/school disjointness verified")


def m_mean(): return DummyRegressor()
def m_ridge(): return make_pipeline(SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True), StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 30)))
def m_ridge_topk(): return make_pipeline(SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True), StandardScaler(), SelectKBest(f_regression, k=TOP_K_FEATURES), RidgeCV(alphas=np.logspace(-2, 4, 30)))
def m_lgbm(): return LGBMRegressor(n_estimators=400, learning_rate=0.02, num_leaves=8, min_child_samples=15, subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=5.0, random_state=SEED, n_jobs=4, verbose=-1)


class TwoStage(BaseEstimator, RegressorMixin):
    """Stage 1: ridge on screening columns. Stage 2: LightGBM on the stage-1 residual, where the
    residual targets are inner out-of-fold, so stage 2 never sees optimistic in-sample residuals."""

    def __init__(self, screen_cols, inner_folds=5):
        self.screen_cols, self.inner_folds = screen_cols, inner_folds

    def fit(self, X, y):
        Xs = X[self.screen_cols]
        self.base_ = m_ridge().fit(Xs, y)
        inner = cross_val_predict(m_ridge(), Xs, y, cv=self.inner_folds)
        self.res_ = m_lgbm().fit(X, y - inner)
        return self

    def predict(self, X):
        return self.base_.predict(X[self.screen_cols]) + self.res_.predict(X)


MODELS = {"mean": m_mean, "ridge": m_ridge, "ridge_topk": m_ridge_topk, "lgbm": m_lgbm,
          "two_stage": lambda: TwoStage(SCREEN)}


def oof_predict(model_name, cols, scheme, target=None):
    target = y if target is None else target
    folds = FOLDS[scheme]; n_rep = 1 + max(r for r, _, _ in folds)
    P = np.full((n_rep, len(TR)), np.nan)
    X = TR[cols]
    for r, a, b in folds:
        m = MODELS[model_name]()
        m.fit(X.iloc[a], target[a])
        P[r, b] = m.predict(X.iloc[b])
    return P


def score(P, target=None):
    target = y if target is None else target
    rm = [np.sqrt(mean_squared_error(target, p)) for p in P]
    r2 = [r2_score(target, p) for p in P]
    return {"rmse": np.mean(rm), "rmse_sd": np.std(rm), "r2": np.mean(r2)}


BASELINES = {"M00_mean": ("mean", SCREEN), "M01_ridge_thetas": ("ridge", THETAS),
             "M02_ridge_screen": ("ridge", SCREEN), "M03_lgbm_screen": ("lgbm", SCREEN)}
OOF = {}
rows = []
for name, (mdl, cols) in BASELINES.items():
    for scheme in FOLDS:
        P = oof_predict(mdl, cols, scheme); OOF[(name, scheme)] = P
        rows.append({"model": name, "scheme": scheme, **score(P)})
baseline_res = save_table(pd.DataFrame(rows), "08_baselines")
display(baseline_res.pivot(index="model", columns="scheme", values=["rmse", "r2"]))

base_pred = OOF[("M02_ridge_screen", "student")].mean(0)
TR["screen_resid"] = y - base_pred
print(f"primary baseline M02 OOF R² = {r2_score(y, base_pred):.3f}; residual SD = {TR.screen_resid.std():.2f}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].scatter(base_pred, y, s=8, alpha=.5); ax[0].plot([y.min(), y.max()], [y.min(), y.max()], "k--", lw=1)
ax[0].set(xlabel="OOF screening prediction", ylabel="outcome", title="M02 screening baseline")
ax[1].hist(TR.screen_resid, bins=40); ax[1].set(title="screening residual")
sr_ = TR.groupby("school_id").screen_resid.agg(["mean", "size"]).sort_values("mean")
ax[2].bar(sr_.index.astype(str), sr_["mean"]); ax[2].set(title="mean residual by school", xlabel="school")
save_fig(fig, "08_baseline")
print("residual by instrument / sample rate:")
display(TR.groupby(["instrument", "sr_11025"]).screen_resid.agg(["mean", "std", "size"]))

## 9 · Residual signal discovery

For every student feature: Spearman correlation with the outcome (raw) and with the screening residual, BH-FDR across all features, cluster (school) bootstrap CIs for the top candidates, and a **split-half replication** in which schools are divided into two disjoint halves — a feature replicates if it has the same sign and p < 0.10 in both halves. These are associations, not causes, and they are EDA: predictive claims come only from the nested CV in §12.

In [ ]:
FEATS = RESP + ACOU
half_rng = np.random.default_rng(SEED + 1)
sch = pd.Series(TR.school_id.unique()); sch = sch.sample(frac=1, random_state=SEED + 1).to_numpy()
cum = TR.school_id.value_counts().reindex(sch).cumsum() / len(TR)
half_a = set(cum.index[cum <= 0.5]) or {sch[0]}
in_a = TR.school_id.isin(half_a).to_numpy()
print(f"split-half: {in_a.mean():.0%} / {(~in_a).mean():.0%} of students")

res = TR.screen_resid.to_numpy()
rows = []
for c in FEATS:
    x = TR[c].to_numpy(float)
    r_raw, p_raw, n = spearman(x, y)
    r_res, p_res, _ = spearman(x, res)
    ra, pa, _ = spearman(x[in_a], res[in_a]); rb, pb, _ = spearman(x[~in_a], res[~in_a])
    rows.append({"feature": c, "family": family_of(c), "task": task_of(c), "n": n, "rho_outcome": r_raw, "p_outcome": p_raw,
                 "rho_resid": r_res, "p_resid": p_res, "rho_half_a": ra, "rho_half_b": rb,
                 "replicated": bool(np.sign(ra) == np.sign(rb) and pa < 0.10 and pb < 0.10)})
disc = pd.DataFrame(rows)
disc["q_outcome"] = bh_fdr(disc.p_outcome)
disc["q_resid"] = bh_fdr(disc.p_resid)


def cluster_boot_rho(x, r, groups, n_boot=N_BOOT, seed=SEED):
    g_rng = np.random.default_rng(seed)
    ug = np.unique(groups); idx = {g: np.flatnonzero(groups == g) for g in ug}
    out = []
    for _ in range(n_boot):
        take = np.concatenate([idx[g] for g in g_rng.choice(ug, len(ug))])
        out.append(spearman(x[take], r[take])[0])
    return np.nanpercentile(out, [2.5, 97.5])


top = disc.sort_values("p_resid").head(40).index
for i in top:
    lo, hi = cluster_boot_rho(TR[disc.at[i, "feature"]].to_numpy(float), res, schools)
    disc.loc[i, ["ci_low", "ci_high"]] = lo, hi
save_table(disc, "09_residual_associations")

print(f"features tested: {len(disc)} | FDR<{FDR_ALPHA} vs outcome: {(disc.q_outcome < FDR_ALPHA).sum()} | vs residual: {(disc.q_resid < FDR_ALPHA).sum()} | replicated: {disc.replicated.sum()}")
display(disc.sort_values("p_resid").head(20)[["feature", "family", "task", "rho_outcome", "rho_resid", "q_resid", "ci_low", "ci_high", "replicated"]])

fam_rank = (disc.groupby("family").agg(n=("feature", "size"), max_abs_rho_resid=("rho_resid", lambda s: s.abs().max()),
                                       n_fdr=("q_resid", lambda q: int((q < FDR_ALPHA).sum())),
                                       n_replicated=("replicated", "sum")).sort_values("max_abs_rho_resid", ascending=False))
display(save_table(fam_rank.reset_index(), "09_family_ranking"))

In [ ]:
t20 = disc.sort_values("p_resid").head(20).iloc[::-1]
colors = {f: f"C{i}" for i, f in enumerate(["response", "temporal", "spectral", "mfcc", "prosody", "wavelet", "quality"])}
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ax[0].barh(t20.feature, t20.rho_resid, color=[colors[f] for f in t20.family])
err = np.vstack([t20.rho_resid - t20.ci_low, t20.ci_high - t20.rho_resid])
ax[0].errorbar(t20.rho_resid, t20.feature, xerr=err, fmt="none", ecolor="k", lw=.8)
ax[0].axvline(0, c="k", lw=.5); ax[0].set(title="top residual associations (school-bootstrap 95% CI)", xlabel="Spearman rho with screening residual")
for f, c in colors.items():
    d = disc[disc.family == f]
    ax[1].scatter(d.rho_outcome, d.rho_resid, s=8, c=c, label=f, alpha=.6)
ax[1].axhline(0, c="k", lw=.5); ax[1].axvline(0, c="k", lw=.5)
ax[1].set(xlabel="rho with outcome", ylabel="rho with screening residual", title="raw vs incremental association"); ax[1].legend(fontsize=8)
save_fig(fig, "09_residual_discovery")

## 10 · Confounding and robustness

Recording conditions can leak outcome information (e.g. a school with one device and a different outcome distribution). Checks:

1. association of each feature with the **original sample rate** (11.025 kHz recordings were upsampled — bandwidth-limited) and with the **instrument**;
2. **school separability** per family — how well a family identifies the school vs. the majority baseline;
3. sign consistency of the top features inside the 16 kHz-only subset;
4. **negative control**: the full model trained on a shuffled outcome must score R² ≈ 0.

In [ ]:
sr_flag = TR.sr_11025.to_numpy(float); inst = TR.wj4.to_numpy(float)
d16 = (TR.sr_11025 == 0).to_numpy()
conf = []
for c in FEATS:
    x = TR[c].to_numpy(float)
    conf.append({"feature": c, "rho_sample_rate": spearman(x, sr_flag)[0], "rho_instrument": spearman(x, inst)[0],
                 "rho_resid_16k_only": spearman(x[d16], res[d16])[0]})
conf = pd.DataFrame(conf)
disc = disc.merge(conf, on="feature")
disc["confound_flag"] = (disc.rho_sample_rate.abs() > CONFOUND_R) | (disc.rho_instrument.abs() > CONFOUND_R)
disc["sign_kept_16k"] = np.sign(disc.rho_resid) == np.sign(disc.rho_resid_16k_only)
print("share of features flagged as sample-rate / instrument confounded, by family:")
display(disc.groupby("family").agg(flagged=("confound_flag", "mean"), max_abs_rho_sr=("rho_sample_rate", lambda s: s.abs().max())))
tt = disc.sort_values("p_resid").head(20)
print(f"top-20 residual features keeping their sign in 16 kHz-only data: {tt.sign_kept_16k.mean():.0%}")

# school separability per family
big = TR.school_id.map(TR.school_id.value_counts()) >= MIN_SCHOOL_SIZE
Xs_y = TR.loc[big, "school_id"].to_numpy()
majority = pd.Series(Xs_y).value_counts(normalize=True).iloc[0]
sep = []
for fam in ["response", "temporal", "spectral", "mfcc", "prosody", "wavelet", "quality"]:
    cols = [c for c in FEATS if family_of(c) == fam and c.startswith(("all__", "r_"))]
    if not cols:
        continue
    clf = make_pipeline(SimpleImputer(strategy="median", keep_empty_features=True), StandardScaler(), PCA(n_components=min(10, len(cols)), random_state=SEED), LogisticRegression(max_iter=2000, C=0.5))
    acc = cross_val_score(clf, TR.loc[big, cols], Xs_y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED)).mean()
    auc = cross_val_score(clf, TR[cols], TR.sr_11025, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring="roc_auc").mean()
    sep.append({"family": fam, "n_features": len(cols), "school_accuracy": acc, "majority_baseline": majority, "sample_rate_auc": auc})
school_sep = save_table(pd.DataFrame(sep), "10_school_separability")
display(school_sep)

In [ ]:
# negative control: full feature set, shuffled outcome (shuffled within school to keep school structure)
perm_y = y.copy()
prng = np.random.default_rng(SEED + 7)
for s_ in np.unique(schools):
    ix = np.flatnonzero(schools == s_); perm_y[ix] = prng.permutation(perm_y[ix])
NEG = {}
for mdl in ["ridge", "lgbm"]:
    P = oof_predict(mdl, SCREEN + RESP + ACOU, "student", target=perm_y)
    NEG[mdl] = float(score(P, perm_y)["r2"])
print("negative control (outcome shuffled within school) OOF R²:", {k: round(v, 3) for k, v in NEG.items()})

## 11 · Student phenotype atlas

Unsupervised clusters on response + overall acoustic features (the outcome is never used). We then ask whether clusters differ in screening residual — i.e. whether "similar screening, different profile" exists. Descriptive only.

In [ ]:
CL_COLS = RESP + [c for c in ACOU if c.startswith("all__")]
Z = make_pipeline(SimpleImputer(strategy="median", keep_empty_features=True), StandardScaler()).fit_transform(TR[CL_COLS])
Zp = PCA(n_components=10, random_state=SEED).fit_transform(Z)
sil = {k: silhouette_score(Zp, KMeans(k, n_init=10, random_state=SEED).fit_predict(Zp)) for k in range(2, 9)}
K = max(sil, key=sil.get)
TR["cluster"] = KMeans(K, n_init=10, random_state=SEED).fit_predict(Zp)
prof = TR.groupby("cluster").agg(n=("subject_id", "size"), outcome=("outcome_score", "mean"), screen_resid=("screen_resid", "mean"),
                                 pct_wj4=("wj4", "mean"), pct_sr11025=("sr_11025", "mean"),
                                 top_school_share=("school_id", lambda s: s.value_counts(normalize=True).iloc[0]))
KW_P = stats.kruskal(*[TR.screen_resid[TR.cluster == k] for k in range(K)]).pvalue
print(f"silhouette by k: { {k: round(v, 3) for k, v in sil.items()} } -> k={K}; Kruskal-Wallis residual p={KW_P:.3g}")
display(save_table(prof.reset_index(), "11_cluster_profiles"))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(Zp[:, 0], Zp[:, 1], c=TR.cluster, cmap="tab10", s=10); ax[0].set(xlabel="PC1", ylabel="PC2", title=f"phenotypes (k={K})")
ax[1].boxplot([TR.screen_resid[TR.cluster == k] for k in range(K)]); ax[1].set(xlabel="cluster", ylabel="screening residual", title=f"residual by cluster (KW p={KW_P:.2g})")
save_fig(fig, "11_clusters")

## 12 · Predictive ablation

Which feature families add predictive value over screening? Every experiment uses the **same folds**, so deltas are paired. Δ RMSE vs the screening baseline (A00, ridge) is bootstrapped over students (student scheme) or over schools (school scheme) — negative Δ = better. `quality` alone is a technical-shortcut control; `acoustic_only` shows what speech carries without any screening.

In [ ]:
fam_cols = lambda fam: [c for c in ACOU if family_of(c) == fam]
SPEECH = [c for c in ACOU if family_of(c) != "quality"]
ABLATIONS = OrderedDict([
    ("A00_screen", SCREEN),
    ("A01_screen+response", SCREEN + RESP),
    ("A02_+temporal", SCREEN + RESP + fam_cols("temporal")),
    ("A03_+spectral", SCREEN + RESP + fam_cols("spectral")),
    ("A04_+mfcc", SCREEN + RESP + fam_cols("mfcc")),
    ("A05_+prosody", SCREEN + RESP + fam_cols("prosody")),
    ("A06_+wavelet", SCREEN + RESP + fam_cols("wavelet")),
    ("A07_+quality(control)", SCREEN + RESP + fam_cols("quality")),
    ("A08_+all_speech", SCREEN + RESP + SPEECH),
    ("A09_everything", SCREEN + RESP + ACOU),
    ("A10_acoustic_only", ACOU),
])
ABL_MODELS = ["ridge", "ridge_topk", "lgbm", "two_stage"]


def boot_delta(e_ref, e_new, groups=None, n_boot=N_BOOT, seed=SEED):
    """95% CI of RMSE(new) - RMSE(ref) from per-student squared errors (paired, cluster-resampled if groups)."""
    b_rng = np.random.default_rng(seed)
    if groups is None:
        idx = [b_rng.integers(0, len(e_ref), len(e_ref)) for _ in range(n_boot)]
    else:
        ug = np.unique(groups); gi = {g: np.flatnonzero(groups == g) for g in ug}
        idx = [np.concatenate([gi[g] for g in b_rng.choice(ug, len(ug))]) for _ in range(n_boot)]
    d = [np.sqrt(e_new[i].mean()) - np.sqrt(e_ref[i].mean()) for i in idx]
    return np.percentile(d, [2.5, 97.5])


t0 = time.time()
rows = []
for scheme in FOLDS:
    ref = OOF[("M02_ridge_screen", scheme)].mean(0)
    e_ref = (y - ref) ** 2
    for name, cols in ABLATIONS.items():
        for mdl in ABL_MODELS:
            if mdl == "two_stage" and not set(SCREEN) <= set(cols):
                continue
            if mdl == "ridge_topk" and len(cols) <= TOP_K_FEATURES:
                continue
            P = oof_predict(mdl, cols, scheme); OOF[(name, mdl, scheme)] = P
            pm = P.mean(0); e = (y - pm) ** 2
            lo, hi = boot_delta(e_ref, e, groups=schools if scheme == "school" else None)
            rows.append({"experiment": name, "model": mdl, "scheme": scheme, "n_features": len(cols), **score(P),
                         "delta_rmse": np.sqrt(e.mean()) - np.sqrt(e_ref.mean()), "ci_low": lo, "ci_high": hi})
    print(f"{scheme} done ({time.time() - t0:.0f}s)")
abl = save_table(pd.DataFrame(rows), "12_ablation")
display(abl.pivot_table(index=["experiment"], columns=["scheme", "model"], values="delta_rmse").round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, scheme in zip(axes, FOLDS):
    d = abl[abl.scheme == scheme]
    exps = list(ABLATIONS)
    for j, mdl in enumerate(ABL_MODELS):
        dm = d[d.model == mdl].set_index("experiment").reindex(exps)
        yy = np.arange(len(exps)) + (j - 1.5) * 0.18
        ax.errorbar(dm.delta_rmse, yy, xerr=[dm.delta_rmse - dm.ci_low, dm.ci_high - dm.delta_rmse], fmt="o", ms=4, label=mdl, capsize=2)
    ax.axvline(0, c="k", lw=.7); ax.set_yticks(range(len(exps))); ax.set_yticklabels(exps)
    ax.set(title=f"Δ RMSE vs screening ridge — {scheme} CV", xlabel="Δ RMSE (negative = better)")
axes[0].legend(fontsize=8); axes[0].invert_yaxis()
save_fig(fig, "12_ablation")

best = abl[abl.scheme == "student"].sort_values("rmse").iloc[0]
print(f"best (student CV): {best.experiment} / {best.model}: RMSE {best.rmse:.3f}, R² {best.r2:.3f}, Δ {best.delta_rmse:+.3f} [{best.ci_low:+.3f}, {best.ci_high:+.3f}]")
bs = abl[(abl.scheme == "school") & (abl.experiment == best.experiment) & (abl.model == best.model)].iloc[0]
print(f"same model under school CV: RMSE {bs.rmse:.3f}, R² {bs.r2:.3f}, Δ {bs.delta_rmse:+.3f} [{bs.ci_low:+.3f}, {bs.ci_high:+.3f}]")

## 13 · Failure atlas

Which students does the best model miss, and what do they share? Large errors at the extremes partly reflect regression to the mean; the table compares the worst 10 % to the rest (standardised mean difference).

In [ ]:
best_key = (best.experiment, best.model, "student")
pred_best = OOF[best_key].mean(0)
TR["oof_best"] = pred_best; TR["err_best"] = y - pred_best
worst = TR.err_best.abs() >= TR.err_best.abs().quantile(0.9)
probe = {"outcome": TR.outcome_score, "abs_screen_resid": TR.screen_resid.abs(), "deletion_theta_missing": TR.deletion_theta.isna().astype(float),
         "wj4": TR.wj4, "sr_11025": TR.sr_11025, "n_items": TR.r_total_n, "n_unscored": TR.r_n_unscored,
         "snr_db": TR.get("all__q_snr_db"), "speech_ratio": TR.get("all__t_speech_ratio"), "initial_silence": TR.get("all__t_initial_silence")}
fr = []
for k, v in probe.items():
    if v is None: continue
    v = v.astype(float); a, b = v[worst], v[~worst]
    fr.append({"variable": k, "worst_10pct": a.mean(), "rest": b.mean(), "smd": (a.mean() - b.mean()) / (v.std() + 1e-9),
               "p": stats.mannwhitneyu(a.dropna(), b.dropna()).pvalue})
fail = save_table(pd.DataFrame(fr).sort_values("smd", key=abs, ascending=False), "13_failure_profile")
display(fail)
print("error by school (worst-first):")
display(TR.groupby("school_id").err_best.agg(rmse=lambda e: np.sqrt((e ** 2).mean()), bias="mean", n="size").sort_values("rmse", ascending=False).head(6))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].scatter(y, TR.err_best, s=8, alpha=.5); ax[0].axhline(0, c="k", lw=.5); ax[0].set(xlabel="outcome", ylabel="OOF error", title="error vs outcome (regression to mean)")
ax[1].scatter(TR.screen_resid, TR.err_best, s=8, alpha=.5); ax[1].set(xlabel="screening residual", ylabel="best-model error", title="what the best model recovers")
save_fig(fig, "13_failures")

## 14 · Feature registry and task relevance

One row per student feature, with separate evidence columns rather than one importance score. Status rules (explicit, conservative):

* **ROBUST** — residual FDR q < α, school-bootstrap CI excludes 0, replicated across school halves, not confound-flagged, sign kept in 16 kHz-only data;
* **PROMISING** — p_resid < 0.05 and (replicated or CI excludes 0), not confounded;
* **CONFOUNDED** — p_resid < 0.05 but flagged for sample-rate / instrument association;
* **NULL** — otherwise.

In [ ]:
reg = disc.copy()
ci_ex0 = (reg.ci_low > 0) | (reg.ci_high < 0)
reg["status"] = "NULL"
reg.loc[(reg.p_resid < 0.05) & (reg.replicated | ci_ex0) & ~reg.confound_flag, "status"] = "PROMISING"
reg.loc[(reg.q_resid < FDR_ALPHA) & ci_ex0 & reg.replicated & ~reg.confound_flag & reg.sign_kept_16k, "status"] = "ROBUST"
reg.loc[(reg.p_resid < 0.05) & reg.confound_flag, "status"] = "CONFOUNDED"
reg = reg.sort_values(["status", "p_resid"], key=lambda s: s.map({"ROBUST": 0, "PROMISING": 1, "CONFOUNDED": 2, "NULL": 3}) if s.name == "status" else s)
save_table(reg, "14_feature_registry")
display(reg.status.value_counts().to_frame())
display(reg[reg.status.isin(["ROBUST", "PROMISING"])].head(25)[["feature", "family", "task", "rho_resid", "q_resid", "ci_low", "ci_high", "replicated", "rho_sample_rate", "status"]])

rel = reg[reg.task != "student"].pivot_table(index="family", columns="task", values="rho_resid", aggfunc=lambda s: s.abs().max())
rel_n = reg[reg.task != "student"].pivot_table(index="family", columns="task", values="status", aggfunc=lambda s: s.isin(["ROBUST", "PROMISING"]).sum())
print("task relevance matrix: max |rho| with screening residual (count of ROBUST/PROMISING in brackets)")
display(rel.round(3).astype(str) + " (" + rel_n.astype(int).astype(str) + ")")
fig, ax = plt.subplots(figsize=(7, 3.8))
im = ax.imshow(rel.to_numpy(), cmap="magma")
ax.set(xticks=range(len(rel.columns)), xticklabels=rel.columns, yticks=range(len(rel)), yticklabels=rel.index, title="task × family: max |rho| with residual")
for (i, j), v in np.ndenumerate(rel.to_numpy()):
    ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="w" if v < np.nanmax(rel.to_numpy()) * .7 else "k")
plt.colorbar(im, ax=ax)
save_fig(fig, "14_task_relevance")

## 15 · Hypothesis registry

Each hypothesis is evaluated programmatically from the artefacts above. **SUPPORTED** for a predictive claim needs the paired Δ RMSE CI entirely below 0 under student CV **and** Δ < 0 under school CV (with at least one model). **INCONCLUSIVE** when the point estimate helps but the CI crosses 0.

In [ ]:
def gain(exp, ref_exp=None):
    """Best model's Δ for `exp` vs screening; if ref_exp is given, Δ relative to ref_exp's best model (same scheme)."""
    out = {}
    for scheme in FOLDS:
        d = abl[(abl.scheme == scheme) & (abl.experiment == exp)].sort_values("rmse").iloc[0]
        if ref_exp is None:
            out[scheme] = (d.delta_rmse, d.ci_low, d.ci_high, d.model)
        else:
            r = abl[(abl.scheme == scheme) & (abl.experiment == ref_exp)].sort_values("rmse").iloc[0]
            e_new = (y - OOF[(exp, d.model, scheme)].mean(0)) ** 2; e_ref = (y - OOF[(ref_exp, r.model, scheme)].mean(0)) ** 2
            lo, hi = boot_delta(e_ref, e_new, groups=schools if scheme == "school" else None)
            out[scheme] = (np.sqrt(e_new.mean()) - np.sqrt(e_ref.mean()), lo, hi, d.model)
    return out


def pred_status(g):
    s, c = g["student"], g["school"]
    if s[2] < 0 and c[0] < 0: return "SUPPORTED"
    if s[0] < 0: return "INCONCLUSIVE"
    return "NOT_SUPPORTED"


def fmt(g):
    return "; ".join(f"{k}: Δ={v[0]:+.3f} [{v[1]:+.3f},{v[2]:+.3f}] ({v[3]})" for k, v in g.items())


H = []
m02 = baseline_res.set_index(["model", "scheme"]).loc[("M02_ridge_screen", "student")]
H.append(("H01", "Screening (thetas + instrument + sample rate) explains a substantial share of outcome (OOF R² > 0.2).",
          f"M02 OOF R² = {m02.r2:.3f}", "SUPPORTED" if m02.r2 > 0.2 else "NOT_SUPPORTED"))
H.append(("H02", "The two outcome instruments have different distributions.", f"Mann-Whitney p={mw.pvalue:.3g}, Levene p={lev.pvalue:.3g}",
          "SUPPORTED" if min(mw.pvalue, lev.pvalue) < 0.05 else "NOT_SUPPORTED"))
H.append(("H03", "Outcome clusters by school (ICC > 0.05).", f"ICC = {ICC_SCHOOL:.3f}", "SUPPORTED" if ICC_SCHOOL > 0.05 else "NOT_SUPPORTED"))
g = gain("A01_screen+response"); H.append(("H04", "Item-response features (accuracy, Rasch ability, items reached) add over screening.", fmt(g), pred_status(g)))
for hid, exp, txt in [("H05", "A02_+temporal", "Temporal / fluency"), ("H06", "A03_+spectral", "Spectral"), ("H07", "A04_+mfcc", "MFCC"),
                      ("H08", "A05_+prosody", "Prosody"), ("H09", "A06_+wavelet", "Wavelet")]:
    g = gain(exp, "A01_screen+response"); H.append((hid, f"{txt} features add over screening + response.", fmt(g), pred_status(g)))
g = gain("A08_+all_speech", "A01_screen+response"); H.append(("H10", "All speech acoustics together add over screening + response.", fmt(g), pred_status(g)))
g_q = gain("A07_+quality(control)", "A01_screen+response"); g_s = gain("A08_+all_speech", "A01_screen+response")
H.append(("H11", "Acoustic gain is not a recording-quality shortcut (speech families beat the quality-only control).",
          f"speech Δ={g_s['student'][0]:+.3f} vs quality Δ={g_q['student'][0]:+.3f} (student CV)",
          "SUPPORTED" if g_s["student"][0] < g_q["student"][0] - 0.02 and g_s["student"][0] < 0 else "NOT_SUPPORTED"))
n_rob = int((reg.status == "ROBUST").sum()); n_rob_ac = int(((reg.status == "ROBUST") & (reg.family != "response")).sum())
H.append(("H12", "At least one acoustic feature carries robust incremental (residual) signal.", f"{n_rob_ac} acoustic ROBUST features ({n_rob} ROBUST in total)",
          "SUPPORTED" if n_rob_ac > 0 else ("INCONCLUSIVE" if ((reg.status == "PROMISING") & (reg.family != "response")).any() else "NOT_SUPPORTED")))
sep_ac = school_sep[school_sep.family != "response"]
H.append(("H13", "Acoustic families identify the school above chance (technical leakage risk).",
          f"max school accuracy {sep_ac.school_accuracy.max():.2f} vs majority {majority:.2f}; max sample-rate AUC {sep_ac.sample_rate_auc.max():.2f}",
          "SUPPORTED" if sep_ac.school_accuracy.max() > majority + 0.1 else "NOT_SUPPORTED"))
H.append(("H14", "Negative control: a shuffled outcome is not predictable (R² ≤ 0.02).", f"R² = {NEG}", "SUPPORTED" if max(NEG.values()) <= 0.02 else "NOT_SUPPORTED"))
H.append(("H15", "Unsupervised phenotypes differ in screening residual.", f"k={K}, Kruskal-Wallis p={KW_P:.3g}", "SUPPORTED" if KW_P < 0.05 else "NOT_SUPPORTED"))
g = gain("A10_acoustic_only"); H.append(("H16", "Speech acoustics alone predict the outcome at least as well as screening.", fmt(g),
                                         "SUPPORTED" if g["student"][0] <= 0 else "NOT_SUPPORTED"))
hyp = save_table(pd.DataFrame(H, columns=["id", "hypothesis", "evidence", "status"]), "15_hypothesis_registry")
with pd.option_context("display.max_colwidth", 140):
    display(hyp)

## 16 · Submission

The best experiment/model under student CV (selection among ~40 candidates gives mild optimism — compare with its school-CV score) is refitted on all training students and applied to test students. Output order and ids follow `submission_format.csv`.

In [ ]:
if MAKE_SUBMISSION:
    cols = ABLATIONS[best.experiment]
    final = MODELS[best.model]().fit(TR[cols], y)
    te = TE.set_index("subject_id").reindex(sub_fmt.subject_id)
    pred = np.clip(final.predict(te[cols].reset_index(drop=True)), y.min() - 10, y.max() + 10)
    sub = sub_fmt.copy(); sub["outcome_score"] = pred
    assert list(sub.subject_id) == list(sub_fmt.subject_id) and sub.outcome_score.notna().all() and len(sub) == len(sub_fmt)
    tag = re.sub(r"[^A-Za-z0-9]+", "_", f"{best.experiment}_{best.model}")
    sub_path = OUT_DIR / f"submission_{tag}.csv"
    sub.to_csv(sub_path, index=False)
    print(f"wrote {sub_path} | {len(sub)} rows | pred mean {pred.mean():.2f} sd {pred.std():.2f} (train mean {y.mean():.2f} sd {y.std():.2f})")
    print(f"test students with audio features: {te[ACOU].notna().any(axis=1).mean():.1%}")
else:
    print("MAKE_SUBMISSION is False — skipped")

## 17 · Summary

In [ ]:
lines = ["# WildSpeech P01 — summary", "",
         f"- train students {len(TR)}, test {len(TE)}, train clips {len(clips_tr)}, clips with audio features {len(cf)}",
         f"- outcome mean {y.mean():.1f} (sd {y.std():.1f}); school ICC {ICC_SCHOOL:.3f}; instrument Mann-Whitney p {mw.pvalue:.3g}",
         f"- screening baseline (M02 ridge) OOF R² {m02.r2:.3f}, RMSE {m02.rmse:.3f}",
         f"- best model: {best.experiment} / {best.model} — student CV RMSE {best.rmse:.3f} (R² {best.r2:.3f}, Δ {best.delta_rmse:+.3f} [{best.ci_low:+.3f}, {best.ci_high:+.3f}]); school CV RMSE {bs.rmse:.3f} (Δ {bs.delta_rmse:+.3f})",
         f"- feature registry: " + ", ".join(f"{k} {v}" for k, v in reg.status.value_counts().items()),
         f"- negative control R²: {', '.join(f'{k} {v:.3f}' for k, v in NEG.items())}", "",
         "## Hypotheses", ""] + [f"- **{r.id}** [{r.status}] {r.hypothesis} — {r.evidence}" for r in hyp.itertuples()] + \
        ["", "## Top residual features", ""] + \
        [f"- {r.feature} ({r.family}, {r.task}): rho_resid {r.rho_resid:+.3f}, q {r.q_resid:.3g}, {r.status}" for r in reg.head(15).itertuples()] + \
        ["", "Associations are observational; predictive claims come from paired, student-/school-disjoint CV only."]
(OUT_DIR / "P01_SUMMARY.md").write_text("\n".join(lines), encoding="utf-8")
display(Markdown("\n".join(lines)))
print("tables :", TAB_DIR.resolve()); print("figures:", FIG_DIR.resolve())